# 1. Prepare GEX data

In [14]:
DROP_NULLS = True
DROP_LOUVEAU = True
SELECT_PRE_TREATMENT = True
SELECT_RNA_SEQ = False

## Some mappings

In [15]:
rna_seq_sources = ['Hugo et al.', 'Kwong et al.', 'Yan et al.']
q_pcr_sources = ['Louveau et al.']
micro_array_sources = ['Long et al.', 'Rizos et al.']

source_map = {
    'doi:10.1016/j.cell.2015.07.061': 'Hugo et al.',
    'doi:10.1172/JCI78954DS1': 'Kwong et al.',
    'doi:10.1158/1078-0432.CCR-18-0720': 'Yan et al.',
    'doi:10.3390/cancers11081203': 'Louveau et al.',
    'doi:10.1038/ncomms6694': 'Long et al.',
    'doi:10.1158/1078-0432.CCR-13-3122': 'Rizos et al.'
}

In [16]:
import polars as pl

gex = pl.read_csv("../dataset/original/gene_expressions.csv")
gex = gex.with_columns(pl.col('source').replace(source_map))
gex

id,creation_datetime,patientID,sample_id,HGNC,GeneID,description,value,temporality,source
i64,str,str,str,str,str,str,f64,str,str
1,"""2025-04-23 23:02:05.503260""","""LM_1""","""LMSAM_1""","""BRAF""",null,null,7.60798,"""pre treatment""","""Louveau et al."""
2,"""2025-04-23 23:02:05.503285""","""LM_1""","""LMSAM_1""","""RAF1""",null,null,16.095204,"""pre treatment""","""Louveau et al."""
3,"""2025-04-23 23:02:05.503298""","""LM_1""","""LMSAM_1""","""ARAF""",null,null,4.1515,"""pre treatment""","""Louveau et al."""
4,"""2025-04-23 23:02:05.503312""","""LM_1""","""LMSAM_1""","""PDGFRB""",null,null,1.199885,"""pre treatment""","""Louveau et al."""
5,"""2025-04-23 23:02:05.503324""","""LM_1""","""LMSAM_1""","""IGF1R""",null,null,5.47246,"""pre treatment""","""Louveau et al."""
…,…,…,…,…,…,…,…,…,…
8641387,"""2025-04-24 00:42:16.629746""","""HL_Shi-40""","""Pt21-DP2""","""ZYG11A""",null,null,0.019542,"""progression""","""Hugo et al."""
8641388,"""2025-04-24 00:42:16.629757""","""HL_Shi-40""","""Pt21-DP2""","""ZYG11B""",null,null,4.04421,"""progression""","""Hugo et al."""
8641389,"""2025-04-24 00:42:16.629768""","""HL_Shi-40""","""Pt21-DP2""","""ZYX""",null,null,85.7967,"""progression""","""Hugo et al."""


## Select only 'pre-treatment', drop Louveau

In [17]:
if SELECT_PRE_TREATMENT == True:
    gex = gex.filter((pl.col('temporality') == 'pre treatment'))
if SELECT_RNA_SEQ == True:
    gex = gex.filter(pl.col('source').is_in(rna_seq_sources))
if DROP_LOUVEAU == True:
    gex = gex.filter(pl.col('source') != 'Louveau et al.')

## Drop useless features

In [18]:
gex = gex.drop(['id', 'creation_datetime', 'GeneID', 'description', 'temporality'])

## Sample is useless if patientID or HGNC is not given

In [19]:
gex.select(pl.all().null_count())

patientID,sample_id,HGNC,value,source
u32,u32,u32,u32,u32
92181,0,207000,0,0


In [20]:
if DROP_NULLS == True:
    gex = gex.drop_nulls(subset=['patientID', 'HGNC'])
    gex.select(pl.all().null_count())
    
gex.select(pl.all().null_count())

patientID,sample_id,HGNC,value,source
u32,u32,u32,u32,u32
0,0,0,0,0


## Add Method column

In [21]:
gex = gex.with_columns(
    pl.when(pl.col('source').is_in(rna_seq_sources))
    .then(pl.lit('RNA-seq'))
    .when(pl.col('source').is_in(micro_array_sources))
    .then(pl.lit('micro-array'))
    .otherwise(pl.lit('qPCR'))
    .alias('Method')
)
gex

patientID,sample_id,HGNC,value,source,Method
str,str,str,f64,str,str
"""YR_5306""","""03660445B""","""NAT2""",0.0,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""ADA""",26.233973,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""CDH2""",1.138609,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""AKT3""",12.692677,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""GAGE12F""",0.0,"""Yan et al.""","""RNA-seq"""
…,…,…,…,…,…
"""HL_Shi-43""","""Pt17-baseline""","""ZYG11A""",0.0625681,"""Hugo et al.""","""RNA-seq"""
"""HL_Shi-43""","""Pt17-baseline""","""ZYG11B""",5.74608,"""Hugo et al.""","""RNA-seq"""
"""HL_Shi-43""","""Pt17-baseline""","""ZYX""",45.907933,"""Hugo et al.""","""RNA-seq"""


## Remove duplicate (sample-gene) rows

In [22]:
dupes_pl = (
    gex
    .filter(pl.len().over(['HGNC', 'sample_id']) > 1)
    .sort(['HGNC', 'sample_id'])
)
print(dupes_pl)

shape: (21_042, 6)
┌───────────┬───────────┬────────┬───────┬──────────────┬─────────┐
│ patientID ┆ sample_id ┆ HGNC   ┆ value ┆ source       ┆ Method  │
│ ---       ┆ ---       ┆ ---    ┆ ---   ┆ ---          ┆ ---     │
│ str       ┆ str       ┆ str    ┆ f64   ┆ str          ┆ str     │
╞═══════════╪═══════════╪════════╪═══════╪══════════════╪═════════╡
│ KC_10     ┆ 10A       ┆ ACE    ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_10     ┆ 10A       ┆ ACE    ┆ 2.84  ┆ Kwong et al. ┆ RNA-seq │
│ KC_12     ┆ 12A       ┆ ACE    ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_12     ┆ 12A       ┆ ACE    ┆ 10.27 ┆ Kwong et al. ┆ RNA-seq │
│ KC_13     ┆ 13A       ┆ ACE    ┆ 18.67 ┆ Kwong et al. ┆ RNA-seq │
│ …         ┆ …         ┆ …      ┆ …     ┆ …            ┆ …       │
│ KC_7      ┆ 7A        ┆ mir-95 ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_7      ┆ 7A        ┆ mir-95 ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_9      ┆ 9A        ┆ mir-95 ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_9      ┆ 9A        ┆ mir

In [23]:
gex = gex.sort('value', descending=True).unique(subset=['HGNC', 'sample_id'], keep='first')

dupes_pl = (
    gex
    .filter(pl.len().over(['HGNC', 'sample_id']) > 1)
    .sort(['HGNC', 'sample_id'])
)
print(dupes_pl)

shape: (0, 6)
┌───────────┬───────────┬──────┬───────┬────────┬────────┐
│ patientID ┆ sample_id ┆ HGNC ┆ value ┆ source ┆ Method │
│ ---       ┆ ---       ┆ ---  ┆ ---   ┆ ---    ┆ ---    │
│ str       ┆ str       ┆ str  ┆ f64   ┆ str    ┆ str    │
╞═══════════╪═══════════╪══════╪═══════╪════════╪════════╡
└───────────┴───────────┴──────┴───────┴────────┴────────┘


## Save pre-processed GEX

In [24]:
print(gex)
gex.write_csv(f'../dataset/created/gex.csv')

shape: (4_162_394, 6)
┌────────────┬─────────────────┬──────────────┬───────────┬──────────────┬─────────────┐
│ patientID  ┆ sample_id       ┆ HGNC         ┆ value     ┆ source       ┆ Method      │
│ ---        ┆ ---             ┆ ---          ┆ ---       ┆ ---          ┆ ---         │
│ str        ┆ str             ┆ str          ┆ f64       ┆ str          ┆ str         │
╞════════════╪═════════════════╪══════════════╪═══════════╪══════════════╪═════════════╡
│ RL_WMD-021 ┆ 8755_035H PreB  ┆ KBTBD10      ┆ 23.37587  ┆ Rizos et al. ┆ micro-array │
│ HL_Shi-15  ┆ Pt1-baseline    ┆ PRKAG2       ┆ 1.74235   ┆ Hugo et al.  ┆ RNA-seq     │
│ LR_WMD-022 ┆ 08766 PreC_014G ┆ LOC100134318 ┆ -4.151606 ┆ Long et al.  ┆ micro-array │
│ YR_2148    ┆ 05320220B       ┆ MEF2C        ┆ 10.563233 ┆ Yan et al.   ┆ RNA-seq     │
│ YR_3708    ┆ 03660598B       ┆ COA8         ┆ 11.098728 ┆ Yan et al.   ┆ RNA-seq     │
│ …          ┆ …               ┆ …            ┆ …         ┆ …            ┆ …           │

## Create GEX_MAT

In [25]:
gex_mat = gex.pivot(on='sample_id', index='HGNC', values='value')
gex_mat

HGNC,8755_035H PreB,Pt1-baseline,08766 PreC_014G,05320220B,03660598B,05320191B,05320410C,05320384B,05320229B,05320452B,45534_072K PreB,22A,05420180C,05320146B,04240151B,85284 PreC,15A,28178 PreB_014D,05320108C,Pt8-baseline,05420145C,45534_085E PreB,05320239B,05320179B,28094 PreB,45413 PreC,05320436B,05320420B,Pt16-baseline,05320143B,28178_018A PreB,05320012B,28115 PreB,05320424B,30230_035L PreB,28702_075A PreB,…,05320232B,Pt15-baseline,27228_012A PreB,05320217B,05320141B,Pt4-baseline,28571 PreB,19A,05320388B,05320011B,03660447B,05320444B,2991 PreB,05320435B,28518_049E PreC,05320188B,05320243B,04240106F,05320328B,56216 PreB,05320353C,05320446B,05320425B,05320093C,Pt5-baseline,05320385B,28139_072I PreB,05320286B,05320003B,05420169C,05320459B,05320234B,05320216B,05420129C,Pt2-baseline,05320205B,05320100B
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""KBTBD10""",23.37587,null,7.854737,null,null,null,null,null,null,null,19.06162,null,null,null,null,90.95329,null,79.68283,null,null,null,19.21063,null,null,51.28411,4.135881,null,null,null,null,106.9183,null,98.55662,null,105.9331,26.34831,…,null,null,101.7835,null,null,null,32.6548,null,null,null,null,null,134.3526,null,49.98767,null,null,null,null,15.41566,null,null,null,null,null,null,37.91576,null,null,null,null,null,null,null,null,null,null
"""PRKAG2""",26.15605,1.74235,57.46408,6.567782,1.470895,6.83657,2.405419,3.474732,4.258639,1.516051,69.32901,4.43,6.709182,5.732353,3.881635,188.6185,3.74,88.48717,6.182627,4.84214,3.949571,81.58801,4.625357,1.401799,41.99645,64.96024,2.575316,6.7937,3.22674,4.683057,89.8541,6.101163,120.6739,4.301835,98.56322,106.7821,…,4.777427,5.60647,55.66375,1.617411,3.676807,2.839925,114.468,1.13,5.022807,3.175095,1.444958,2.013216,218.6596,3.37032,119.352,3.025936,4.491322,2.482976,2.552356,80.1801,4.699923,2.023923,4.899994,1.662278,1.784205,3.697752,59.3727,0.894287,5.419454,2.272016,2.202748,3.296798,2.145484,1.692718,3.843775,6.949114,4.202007
"""LOC100134318""",14.48026,null,-4.151606,null,null,null,null,null,null,null,1.137569,null,null,null,null,-1.55382,null,-3.558318,null,null,null,0.6262506,null,null,-4.096674,0.8265495,null,null,null,null,1.592762,null,-1.574794,null,0.1112748,-5.176065,…,null,null,-2.443199,null,null,null,4.463894,null,null,null,null,null,-1.019337,null,-6.337305,null,null,null,null,4.378972,null,null,null,null,null,null,-2.277647,null,null,null,null,null,null,null,null,null,null
"""MEF2C""",69.05501,3.237085,207.1043,10.563233,9.424791,12.539651,9.313326,4.851651,6.747093,2.225821,149.7639,13.11,8.545186,1.82088,17.041539,243.6076,4.71,102.3737,8.601241,7.94673,2.713885,153.655,12.832642,6.184497,326.6496,147.8407,39.676007,18.051391,2.821777,11.313963,121.5717,4.43505,78.71807,8.032211,145.3985,98.41363,…,11.076873,6.040915,144.7731,5.318919,7.751556,4.505045,149.068,6.99,5.85835,16.92061,3.785541,17.939899,121.8676,4.614833,96.87653,16.962204,14.056876,8.929444,2.641103,250.4636,13.780271,21.674343,5.672496,8.44168,5.462175,11.05986,103.1068,14.181716,3.989293,18.834454,4.102868,6.859104,2.735239,7.545793,9.756905,13.143111,8.339831
"""COA8""",null,null,null,9.476742,11.098728,4.790049,10.364591,14.121022,11.451292,6.076787,null,null,9.789443,13.349804,6.611955,null,null,null,7.147636,null,10.129414,null,10.325629,12.786499,null,null,10.527135,9.768661,null,12.298715,null,15.299231,null,8.626133,null,null,…,11.313037,null,null,11.460709,17.340215,null,null,null,11.824468,8.623827,12.482483,19.449319,null,9.419433,null,13.898239,20.880874,10.595406,19.896039,null,10.482675,15.716845,10.082713,10.101466,null,12.47295,null,10.73952,17.714397,6.371697,12.045104,14.363797,14.914722,7.780042,null,8.42633,15.368943
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,

In [26]:
gex_mat.write_csv(f'../dataset/created/gex_mat.csv')